# Regression from Scratch v4 (NumPy only) — Income, Neural Additive Model + robust head

**Version:** 4th Version. **Role:** primary deliverable. Pure-NumPy twin of `regression_pytorch_v4.ipynb` — no torch. `sklearn` ONLY for split/scaler/metrics.

**Parity promise:** same split/seed/scalers, same 13 subnets, same He init, same Adam(1e-3)+cosine+wd(1e-4). Hand-coded subnet forward/backward (einsum), Huber backward `e/B` if `|e|<=delta` (tie -> quadratic, like torch) else `delta*sign(e)/B`, best-val-RMSE checkpoint on own stream. Weights → `./weights_v4/regression_scratch.npz`.

In [1]:
import json, os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

assert os.path.exists("weights_v4/regression_pytorch_history.csv"), "oracle must run first (parity cell)"
SEED = 42
np.random.seed(SEED)
os.makedirs("weights_v4", exist_ok=True)
DELTA, HH, NF = 0.5, 64, 13

df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims | NAM h={HH} delta={DELTA} | log1p target, outlier kept")

X_temp, X_test_raw, y_temp_raw, y_test_raw = train_test_split(X_full, y_raw, test_size=0.20, random_state=SEED)
scaler = StandardScaler().fit(X_temp)
Xtr = scaler.transform(X_temp).astype(np.float64); Xte = scaler.transform(X_test_raw).astype(np.float64)
y_temp_log = np.log1p(y_temp_raw).reshape(-1, 1); y_test_log = np.log1p(y_test_raw).reshape(-1, 1)
y_scaler = StandardScaler().fit(y_temp_log)
ytr_z = y_scaler.transform(y_temp_log).ravel().astype(np.float64)
yte_z = y_scaler.transform(y_test_log).ravel().astype(np.float64)
MU, SIG = float(y_scaler.mean_[0]), float(y_scaler.scale_[0])
print(f"train {Xtr.shape} test {Xte.shape} | y_log scaler: mean {MU:.4f} scale {SIG:.4f}")

P_ = {}
P_["W1"] = np.random.randn(NF, 1, HH) * np.sqrt(2.0 / 1.0); P_["b1"] = np.zeros((NF, HH))
P_["W2"] = np.random.randn(NF, HH, 1) * np.sqrt(2.0 / HH); P_["b2"] = np.zeros((NF, 1))
P_["b0"] = np.zeros(1)
print(f"scratch params: {sum(v.size for v in P_.values())}")

def fwd(Xb):
    B = Xb.shape[0]
    Z1 = Xb[:, :, None] * P_["W1"][:, 0, :][None, :, :] + P_["b1"][None, :, :]  # (B,F,H)
    A1 = np.maximum(Z1, 0)
    Fj = np.einsum("bfh,fh->bf", A1, P_["W2"][:, :, 0]) + P_["b2"].ravel()  # (B,F)
    pred = Fj.sum(1, keepdims=True) + P_["b0"]
    return pred, (Xb, Z1, A1, Fj)

def huber(pred, y):
    e = (pred.ravel() - y.ravel()); B = len(e)
    q = np.abs(e) <= DELTA  # tie -> quadratic branch (torch parity)
    loss = np.where(q, 0.5 * e ** 2, DELTA * (np.abs(e) - 0.5 * DELTA)).mean()
    dp = np.where(q, e, DELTA * np.sign(e)).reshape(-1, 1) / B
    return float(loss), dp

def bwd(dp, cache, G):
    Xb, Z1, A1, Fj = cache; B = Xb.shape[0]
    G["b0"] += dp.sum(0)
    dF = np.broadcast_to(dp, (B, NF))                       # dF[b,f]
    G["b2"] += dF.sum(0).reshape(NF, 1)
    G["W2"] += np.einsum("bfh,bf->fh", A1, dF)[:, :, None]  # dW2[f,h,0] = sum_b A1*dF
    dA = dF[:, :, None] * P_["W2"][None, :, :, 0]            # dA[b,f,h] = dF*W2
    dZ = dA * (Z1 > 0)
    G["b1"] += dZ.sum(0)
    G["W1"] += np.einsum("bf,bfh->fh", Xb, dZ)[:, None, :]



features: 13 dims | NAM h=64 delta=0.5 | log1p target, outlier kept
train (1770, 13) test (443, 13) | y_log scaler: mean 10.7459 scale 0.5137
scratch params: 2510


## numeric grad-check on W2[0,0,0] (2 samples) ----

In [2]:
Xc, yc = Xtr[:2], ytr_z[:2]
p0, c0 = fwd(Xc); l0, _ = huber(p0, yc)
e = 1e-5; old = P_["W2"][0, 0, 0]
P_["W2"][0, 0, 0] = old + e; lp, _ = huber(fwd(Xc)[0], yc)
P_["W2"][0, 0, 0] = old - e; lm, _ = huber(fwd(Xc)[0], yc)
P_["W2"][0, 0, 0] = old
num = (lp - lm) / (2 * e)
_, dp0 = huber(p0, yc); G0 = {k: np.zeros_like(v) for k, v in P_.items()}; bwd(dp0, c0, G0)
ana = G0["W2"][0, 0, 0]
print(f"grad-check W2[0,0,0]: numeric {num:.6e} analytic {ana:.6e} | relerr {abs(num-ana)/(abs(num)+abs(ana)+1e-12):.2e}")
assert abs(num - ana) / (abs(num) + abs(ana) + 1e-12) < 1e-4, "grad-check FAILED"

M = {k: np.zeros_like(v) for k, v in P_.items()}
V = {k: np.zeros_like(v) for k, v in P_.items()}
LR0, WD, B1, B2, EP = 1e-3, 1e-4, 0.9, 0.999, 1e-8

def to_dollar(z): return np.expm1(np.asarray(z).ravel() * SIG + MU)
def eval_rmse():
    pred, _ = fwd(Xte); d = to_dollar(pred)
    return float(np.sqrt(mean_squared_error(y_test_raw, d))), float(mean_absolute_error(y_test_raw, d)), float(r2_score(y_test_raw, d))
def train_huber():
    tot, n = 0.0, 0
    for i in range(0, len(Xtr), 256):
        p, _ = fwd(Xtr[i:i+256]); l, _ = huber(p, ytr_z[i:i+256]); tot += l * (min(i + 256, len(Xtr)) - i); n = len(Xtr)
    return tot / n

history = {"epoch": [], "train_loss": [], "val_rmse": [], "val_r2": [], "lr": []}
best_rmse, best_P = float("inf"), None
N = len(Xtr); EPOCHS = 100; step = 0
for epoch in range(1, EPOCHS + 1):
    lr = 0.5 * LR0 * (1 + np.cos(np.pi * (epoch - 1) / 100))
    perm = np.random.permutation(N)
    for i in range(0, N, 64):
        idx = perm[i:i+64]
        pred, cache = fwd(Xtr[idx]); _, dp = huber(pred, ytr_z[idx])
        G = {k: np.zeros_like(v) for k, v in P_.items()}
        bwd(dp, cache, G); step += 1
        for k in P_:
            g = G[k] + WD * P_[k]
            M[k] = B1 * M[k] + (1 - B1) * g; V[k] = B2 * V[k] + (1 - B2) * (g ** 2)
            mh = M[k] / (1 - B1 ** step); vh = V[k] / (1 - B2 ** step)
            P_[k] -= lr * (mh / (np.sqrt(vh) + EP))
    rmse, mae, r2 = eval_rmse(); tl = train_huber()
    history["epoch"].append(epoch); history["train_loss"].append(tl)
    history["val_rmse"].append(rmse); history["val_r2"].append(r2); history["lr"].append(lr)
    if rmse < best_rmse: best_rmse = rmse; best_P = {k: v.copy() for k, v in P_.items()}
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:03d} | train-huber {tl:.4f} | val-RMSE ${rmse:,.0f} | R2 {r2:.4f} | lr {lr:.2e} | bestRMSE ${best_rmse:,.0f}")
P_ = best_P
print(f"best val-RMSE checkpoint restored (${best_rmse:,.0f})")

np.savez("weights_v4/regression_scratch.npz", **{k: v for k, v in P_.items()},
         scaler_mean=scaler.mean_, scaler_scale=scaler.scale_, columns=np.array(REG_FEATURES),
         y_mean=y_scaler.mean_, y_scale=y_scaler.scale_)
pd.DataFrame(history).to_csv("weights_v4/regression_scratch_history.csv", index=False)
pred, _ = fwd(Xte); d = to_dollar(pred)
mse = float(mean_squared_error(y_test_raw, d))
metrics = {"test_MSE": round(mse, 1), "RMSE": round(float(np.sqrt(mse)), 1),
    "MAE": round(float(mean_absolute_error(y_test_raw, d)), 1),
    "R2": round(float(r2_score(y_test_raw, d)), 4),
    "n_test": int(len(y_test_raw)), "seed": SEED, "delta": DELTA, "target": "log1p+Huber"}
print(json.dumps(metrics, indent=2))
with open("weights_v4/regression_scratch_test_metrics.json", "w") as f: json.dump(metrics, f, indent=2)
ora = json.load(open("weights_v4/regression_pytorch_test_metrics.json"))
rel = abs(metrics["RMSE"] - ora["RMSE"]) / ora["RMSE"]; dr2 = abs(metrics["R2"] - ora["R2"])
print(f"parity: relRMSE {rel:.4f} (<0.10 {'PASS' if rel < 0.10 else 'FAIL'}) | dR2 {dr2:.4f} (<0.05 {'PASS' if dr2 < 0.05 else 'FAIL'})")


grad-check W2[0,0,0]: numeric 1.399970e-01 analytic 1.399970e-01 | relerr 3.48e-12
epoch 001 | train-huber 0.6671 | val-RMSE $458,512 | R2 -454.2112 | lr 1.00e-03 | bestRMSE $458,512
epoch 010 | train-huber 0.0939 | val-RMSE $12,056 | R2 0.6853 | lr 9.80e-04 | bestRMSE $12,056


epoch 020 | train-huber 0.1011 | val-RMSE $13,440 | R2 0.6089 | lr 9.14e-04 | bestRMSE $12,056
epoch 030 | train-huber 0.0909 | val-RMSE $12,036 | R2 0.6863 | lr 8.06e-04 | bestRMSE $11,603


epoch 040 | train-huber 0.0882 | val-RMSE $11,760 | R2 0.7006 | lr 6.69e-04 | bestRMSE $11,533
epoch 050 | train-huber 0.0905 | val-RMSE $12,101 | R2 0.6829 | lr 5.16e-04 | bestRMSE $11,355


epoch 060 | train-huber 0.0871 | val-RMSE $11,412 | R2 0.7180 | lr 3.61e-04 | bestRMSE $11,355
epoch 070 | train-huber 0.0881 | val-RMSE $11,419 | R2 0.7177 | lr 2.19e-04 | bestRMSE $11,355


epoch 080 | train-huber 0.0862 | val-RMSE $11,530 | R2 0.7122 | lr 1.05e-04 | bestRMSE $11,355
epoch 090 | train-huber 0.0859 | val-RMSE $11,498 | R2 0.7137 | lr 2.96e-05 | bestRMSE $11,344


epoch 100 | train-huber 0.0859 | val-RMSE $11,494 | R2 0.7139 | lr 2.47e-07 | bestRMSE $11,344
best val-RMSE checkpoint restored ($11,344)
{
  "test_MSE": 128681375.2,
  "RMSE": 11343.8,
  "MAE": 7055.5,
  "R2": 0.7214,
  "n_test": 443,
  "seed": 42,
  "delta": 0.5,
  "target": "log1p+Huber"
}
parity: relRMSE 0.0032 (<0.10 PASS) | dR2 0.0018 (<0.05 PASS)
